# COMP8851 — T-Finance and T-Social (single file)

Everything needed is inside this notebook. Nothing to attach.

## Three steps

1. **Settings → Accelerator → GPU T4 x2**
2. **Settings → Internet → On**
3. **Run All**

That's it. The notebook unpacks the pipeline, downloads the graphs, decodes
them, trains, and writes the results archive.

## What it runs

| | |
|---|---|
| Models | CARE-GNN, GHRN |
| Datasets | T-Finance, then T-Social |
| Seeds | 2 first for a quick result, then 42 and 72 |
| Epochs | 100, patience 20 — the protocol budget, unchanged |
| Tuning | 2 trials × 10 epochs (protocol ceiling is 12, so this is compliant) |

**Why the tuning is cut and the training is not.** The search is where the time
goes; the final runs are what your reported numbers come from. Trimming the
search costs nothing you report. Trimming the final budget would make these
cells incomparable with the four already completed, which is a gap your
reviewer already raised once.

The graphs are used whole. A subsampled T-Social is not T-Social, so the
result would not be a T-Social number.

## What to expect

T-Social used about **34 GB of GPU memory** on an A6000. A Kaggle T4 has
**16 GB**. T-Finance should complete for both models; T-Social may run out of
memory. Its results are archived before T-Social is attempted, so a failure
there cannot cost you the cells that worked, and an out-of-memory failure is
recorded with its traceback as a real outcome.


In [ ]:
# ---------------------------------------------------------------- settings
FAST_FIRST = True     # seed 2 alone first, so a real result exists early
ALL_SEEDS  = True     # then 42 and 72
DO_TSOCIAL = True     # attempt T-Social after T-Finance is saved

TRIALS, TUNE_EPOCHS, TUNE_PATIENCE = 2, 10, 5
FINAL_EPOCHS, FINAL_PATIENCE = 100, 20      # protocol budget - leave alone

# Label ratios for the final runs. TR40 alone is the fast path.
#
# ["TR40", "TR30", "TR20", "TR10"] additionally produces the label-scarcity
# curve the protocol requires (v4.4 s4.9). Tuning still happens once at TR40
# and the frozen configuration is reused unchanged at every lower ratio, so
# this costs three extra FINAL runs per seed and no extra searching. The lower
# ratios train on less data, so each is cheaper than the TR40 run itself -
# roughly +2x wall clock for the whole sweep, not +4x.
RATIOS = ["TR40"]
# ---------------------------------------------------------------------------

import base64, io, os, shutil, subprocess, sys, tarfile, time, traceback, json
from pathlib import Path

WORK = Path("/kaggle/working/comp8851")
OUTDIR = Path("/kaggle/working")
WORK.mkdir(parents=True, exist_ok=True)

STATUS = {}

def step(name):
    def wrap(fn):
        print(f"\n{'='*70}\n{name}\n{'='*70}")
        t0 = time.time()
        try:
            out = fn()
            STATUS[name] = {"ok": True, "minutes": (time.time()-t0)/60}
            return out
        except Exception as exc:
            STATUS[name] = {"ok": False, "error": f"{type(exc).__name__}: {exc}",
                            "minutes": (time.time()-t0)/60}
            print(f"FAILED: {type(exc).__name__}: {exc}")
            traceback.print_exc(limit=3)
            return None
    return wrap

import torch
GPU_GB = 0.0
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    GPU_GB = p.total_memory / 1024**3
    print(f"GPU   : {p.name}  ({GPU_GB:.1f} GB)")
else:
    print("NO GPU. Settings -> Accelerator -> GPU T4 x2, then Run All again.")
print(f"torch : {torch.__version__}")
print(f"python: {sys.version.split()[0]}")

## 1. Unpack the pipeline

The training code is embedded in the next cell as a compressed archive. It is
written to `/kaggle/working/comp8851` and used from there.

In [ ]:
PIPELINE = (
    "..."
    "f3LV+PT59Pn0+fT5X/L5H5IzHqIA0AcA"
)

@step("1. unpack pipeline")
def _():
    raw = base64.b64decode("".join(PIPELINE))
    with tarfile.open(fileobj=io.BytesIO(raw), mode="r:gz") as tar:
        # filter="data" is the safe extraction mode and becomes the default in
        # Python 3.14; passing it explicitly silences the deprecation warning
        # and keeps behaviour identical across versions.
        try:
            tar.extractall(WORK, filter="data")
        except TypeError:            # Python < 3.12 has no filter argument
            tar.extractall(WORK)
    files = sorted(WORK.rglob("*.py"))
    print(f"   {len(files)} python files written to {WORK}")
    for need in ("scripts/run_benchmark.py", "shared/comp8851/fetch_datasets.py",
                 "models/care-gnn/scripts/run_one.py",
                 "models/ghrn/scripts/run_one.py"):
        print(f"   {'OK  ' if (WORK/need).exists() else 'MISSING'}  {need}")
    (WORK / "data").mkdir(exist_ok=True)
    os.chdir(WORK)
    return True

## 2. Dependencies

Kaggle's PyTorch usually has no matching DGL wheel, and T-Finance and T-Social
ship **as DGL binary graphs**. The fast path installs a matching DGL if one
exists; otherwise the pipeline builds an isolated decoder environment during
the freeze step and training proceeds from the decoded `.npz` with no DGL at
all.

In [ ]:
@step("2. internet check")
def _():
    """Fail in seconds if the Internet toggle is off, not seven minutes in.

    Kaggle's Internet switch is separate from the accelerator, and with it off
    every pip index and the dataset download fail with DNS errors that look
    like unrelated problems.
    """
    import socket
    for host in ("pypi.org", "drive.google.com", "data.dgl.ai"):
        try:
            socket.setdefaulttimeout(8)
            socket.gethostbyname(host)
            print(f"   OK    {host}")
        except OSError as error:
            print(f"   FAIL  {host}: {error}")
            print()
            print("   " + "!" * 62)
            print("   INTERNET IS OFF. Nothing below can work.")
            print("   Settings -> Internet -> On, then Run All again.")
            print("   It is a separate switch from the GPU, and Kaggle may ask")
            print("   you to verify a phone number before it can be enabled.")
            print("   " + "!" * 62)
            raise SystemExit("Internet disabled - enable it and re-run")
    return True


TORCH_PIN = "2.4.0"        # DGL 2.4.0 links against this exact build
DGL_PIN = "2.4.0+cu121"


@step("2b. torch + DGL")
def _():
    """Pin torch to the build DGL is compiled against, in THIS environment.

    Kaggle ships a torch far newer than any DGL wheel, and T-Finance and
    T-Social are DGL binaries. Rather than build a separate decoder
    environment - which depends on ensurepip, and Kaggle's is broken - this
    aligns the main environment with the pairing already proven on this
    project's Vast.ai runs: torch 2.4.0+cu121 with dgl 2.4.0+cu121, both cp312,
    the same as Kaggle.

    GPU training is preserved because the torch build is the CUDA 12.1 one, not
    CPU-only. A T4 runs cu121 natively.
    """
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=False)

    def probe():
        r = subprocess.run(
            [sys.executable, "-c",
             "import dgl, torch; g = dgl.graph(([0],[1]));"
             " print(dgl.__version__, torch.__version__, torch.cuda.is_available())"],
            capture_output=True, text=True)
        return r.stdout.strip() if r.returncode == 0 else None

    ready = probe()
    if ready and ready.split()[1].startswith(TORCH_PIN):
        print(f"   already aligned: {ready}")
        return True

    print(f"   installing torch {TORCH_PIN}+cu121  (~2.5 GB, a few minutes)")
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"torch=={TORCH_PIN}",
         "--index-url", "https://download.pytorch.org/whl/cu121"],
        capture_output=True, text=True)
    if r.returncode != 0:
        print(f"   torch install failed: {(r.stderr or r.stdout)[-300:]}")

    print(f"   installing dgl {DGL_PIN}")
    for index in ("https://data.dgl.ai/wheels/torch-2.4/cu121/repo.html",
                  "https://data.dgl.ai/wheels/torch-2.4/repo.html"):
        r = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", f"dgl=={DGL_PIN}",
             "-f", index], capture_output=True, text=True)
        if probe():
            break
        # Some mirrors serve the version without the local +cu121 tag.
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "dgl==2.4.0",
                        "-f", index], capture_output=True, text=True)
        if probe():
            break

    ready = probe()
    if ready:
        dgl_v, torch_v, cuda = ready.split()
        print(f"   OK  dgl {dgl_v} | torch {torch_v} | cuda available: {cuda}")
        if cuda != "True":
            print("   WARNING: torch reports no CUDA. Training will be slow.")
        print()
        print("   >>> RESTART REQUIRED <<<")
        print("   Torch was replaced in a running kernel, so the already-imported")
        print("   module is stale. Run -> Restart & Run All once. This step will")
        print("   then see the new versions and skip straight past.")
        return True

    print("   Could not align torch and DGL here.")
    print("   Falling back to the isolated decoder in the next step.")
    return False


@step("2c. build the isolated decoder (only if needed)")
def _():
    """Build the decoder now, with output, rather than silently mid-freeze.

    It installs its own torch plus a matching DGL, so it downloads roughly
    2.5 GB and takes several minutes. Doing it here means the progress is
    visible instead of looking like a hung freeze step.
    """
    sys.path.insert(0, str(WORK))
    from shared.comp8851 import dgl_env

    if dgl_env.dgl_importable():
        print("   main interpreter reads DGL; no decoder needed")
        return True

    print("   Building the decoder environment.")
    print("   Expect ~2.5 GB of downloads and roughly 3-8 minutes.")
    print("   It is not stuck - pip is quiet while it resolves wheels.\n")
    started = time.time()
    python, message = dgl_env.create_environment(WORK / ".dgl_env", verbose=True)
    print(f"\n   {message}  ({(time.time()-started)/60:.1f} min)")
    if python is None:
        print("   The decoder could not be built on this host. T-Finance and")
        print("   T-Social cannot be decoded here; the freeze step will report")
        print("   them as blocked rather than failing silently.")
        return False
    return True

## 3. Download and decode

T-Social is ~744 MB compressed and ~4.1 GB extracted. Archives are deleted
afterwards to reclaim disk. Each dataset is frozen separately so a failure on
one cannot cost the other.

In [ ]:
@step("3a. download")
def _():
    import gdown
    gdown.download_folder(
        "https://drive.google.com/drive/folders/1PpNwvZx_YRSCDiHaBUmRIS3x1rZR7fMr",
        output=str(WORK / "data/bwgnn_drive"), quiet=False, use_cookies=False)

    for name in (["tfinance", "tsocial"] if DO_TSOCIAL else ["tfinance"]):
        src = WORK / f"data/bwgnn_drive/dataset/{name}.zip"
        dst = WORK / "data" / name
        dst.mkdir(parents=True, exist_ok=True)
        if not src.exists():
            print(f"   {name}: archive missing")
            continue
        subprocess.run(["unzip", "-q", "-o", str(src), "-d", str(dst)], check=False)
        found = [p for p in dst.rglob(name) if p.is_file()]
        if found and found[0] != dst / name:
            shutil.move(str(found[0]), str(dst / name))
        target = dst / name
        print(f"   {name}: {target.stat().st_size/1024**2:.0f} MB"
              if target.exists() else f"   {name}: NOT FOUND")
    for z in (WORK / "data/bwgnn_drive").rglob("*.zip"):
        z.unlink()
    subprocess.run(["df", "-h", "/kaggle/working"])
    return True


def freeze(name):
    r = subprocess.run([sys.executable, "shared/comp8851/fetch_datasets.py",
                        "--data-root", "data", "--freeze", "--only", name],
                       capture_output=True, text=True, cwd=str(WORK))
    for line in (r.stdout or "").strip().splitlines()[-10:]:
        print("   ", line)
    cache = list((WORK / "data" / name).glob("*_canonical.npz"))
    print(f"   -> {'OK ' + cache[0].name if cache else 'FAILED'}")
    return bool(cache)

tfinance_ok = step("3b. decode T-Finance")(lambda: freeze("tfinance")) or False
tsocial_ok = (step("3c. decode T-Social")(lambda: freeze("tsocial")) or False) if DO_TSOCIAL else False

## 4. Train

Seed 2 runs first so a genuine result exists early. Seeds 42 and 72 follow.

In [ ]:
def run_pair(model, dataset, seeds, trials, minutes, extra=None):
    if not (WORK / "data" / dataset).exists():
        print(f"   SKIP {model} x {dataset}: dataset absent")
        return False
    cmd = [sys.executable, "scripts/run_benchmark.py",
           "--models", model, "--datasets", dataset,
           "--ratios", *RATIOS, "--seeds", *[str(s) for s in seeds],
           "--trials", str(trials),
           "--tune-epochs", str(TUNE_EPOCHS), "--tune-patience", str(TUNE_PATIENCE),
           "--epochs", str(FINAL_EPOCHS), "--patience", str(FINAL_PATIENCE),
           "--max-minutes", str(minutes),
           "--data-root", "data", "--results-root", "results",
           "--no-archive", "--no-report"] + (extra or [])
    proc = subprocess.Popen(cmd, cwd=str(WORK), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    keep = ("trial", "COMPLETE", "FAILED", "seed", "isolation", "Error",
            "error", "memory", "Traceback")
    for line in proc.stdout:
        if any(k in line for k in keep):
            print("   ", line.rstrip())
    proc.wait()
    print(f"   exit {proc.returncode}")
    return proc.returncode == 0


def train(dataset, ok, minutes, extra_by_model=None, order=("CARE-GNN", "GHRN")):
    if not ok:
        print(f"   {dataset} unavailable; skipping")
        return
    for model in order:
        extra = (extra_by_model or {}).get(model)
        if FAST_FIRST:
            step(f"{model} x {dataset} (seed 2)")(
                lambda m=model, d=dataset, e=extra:
                run_pair(m, d, [2], TRIALS, minutes, e))
        if ALL_SEEDS:
            step(f"{model} x {dataset} (seeds 42, 72)")(
                lambda m=model, d=dataset, e=extra:
                run_pair(m, d, [2, 42, 72], TRIALS, minutes, e))

train("tfinance", tfinance_ok, 45)

## 5. Save T-Finance now

Before T-Social is attempted. If that run exhausts memory and takes the kernel
with it, this archive already exists.

In [ ]:
@step("5. archive T-Finance")
def _():
    if not (WORK / "results").exists():
        print("   no results yet")
        return False
    shutil.make_archive(str(OUTDIR / "tfinance_results"), "zip", str(WORK / "results"))
    mb = (OUTDIR / "tfinance_results.zip").stat().st_size / 1024**2
    print(f"   saved tfinance_results.zip ({mb:.1f} MB)")
    print("   Download it from the Output panel on the right.")
    return True

## 6. T-Social

Attempted rather than assumed. GHRN first — its sparse backbone has the better
chance. CARE-GNN needs `--allow-large` to pass its node guard and is unlikely
to fit Kaggle's RAM, but the protocol says attempt every pair and record what
happens. **A failure here is a result.**

In [ ]:
if tsocial_ok:
    print(f"GPU has {GPU_GB:.1f} GB; T-Social used ~34 GB on an A6000.")
    if GPU_GB < 30:
        print("Attempting anyway so the outcome is measured, not assumed.\n")
    # GHRN first, deliberately. Its sparse backbone handles this graph; CARE-GNN
    # materialises per-relation Python adjacency sets over 5.78M nodes and was
    # measured at roughly 11 minutes per epoch here, so it cannot finish 100
    # epochs in any session and will consume the whole budget trying. Running it
    # first once left GHRN with no time at all.
    train("tsocial", tsocial_ok, 180,
          extra_by_model={"CARE-GNN": ["--allow-large"]},
          order=("GHRN", "CARE-GNN"))
elif DO_TSOCIAL:
    print("T-Social did not decode - most likely RAM or disk during the freeze.")
    print("That is a recordable blocked outcome, not a silent gap.")

## 7. Package and report

In [ ]:
@step("7. package")
def _():
    if not (WORK / "results").exists():
        print("   No results directory: nothing trained, so nothing to package.")
        print("   Check the earlier steps - the first FAIL in the summary below")
        print("   is the one to fix.")
        return False
    r = subprocess.run([sys.executable, "scripts/build_deliverable.py",
                        "--results", "results",
                        "--out", str(OUTDIR / "DELIVERABLE_KAGGLE"),
                        "--platform", "Kaggle (Tesla T4)"],
                       capture_output=True, text=True, cwd=str(WORK))
    print(r.stdout[-2000:])
    shutil.make_archive(str(OUTDIR / "tfinance_tsocial_results"), "zip",
                        str(WORK / "results"))
    return True

print(f"\n{'='*70}\nRUN SUMMARY\n{'='*70}")
for name, info in STATUS.items():
    mark = "OK  " if info["ok"] else "FAIL"
    detail = "" if info["ok"] else f"  {info['error'][:60]}"
    print(f"  {mark}  {name:<38} {info['minutes']:5.1f} min{detail}")

cells_done = sorted({
    (json.loads(p.read_text())["model"], json.loads(p.read_text())["dataset"],
     json.loads(p.read_text())["train_seed"])
    for p in (WORK / "results").rglob("summary.json")
    if (WORK / "results").exists()
    and json.loads(p.read_text()).get("test_metrics")
    and not str(json.loads(p.read_text())["configuration"].get("run_mode","")).startswith("tuning")
}) if (WORK / "results").exists() else []

print(f"\n{'='*70}\nCOMPLETED MODEL x DATASET x SEED\n{'='*70}")
for model, dataset, seed in cells_done:
    print(f"  {model:<9} {dataset:<10} seed {seed}")
print(f"  total: {len(cells_done)} genuine test evaluations")

print(f"\n{'='*70}\nDOWNLOAD THESE (Output panel, right)\n{'='*70}")
for f in sorted(OUTDIR.glob("*.zip")):
    print(f"  {f.name:<38} {f.stat().st_size/1024**2:8.1f} MB")